In [ ]:
!pip install statsmodels
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# 1. Load datasets
footfall = pd.read_csv("StationFootfall_2022_2025.csv")
matches = pd.read_excel("wembley_matches_2021_2025.xlsx")
concerts = pd.read_csv("wembley_concerts_2022_2025.csv")
journeys = pd.read_csv("Journeys_2024_2025 .csv")

# 2. Filter Wembley stations
wembley_stations = ["Wembley Park", "Wembley Central", "North Wembley"]
wembley = footfall[footfall["Station"].isin(wembley_stations)].copy()

# Convert date and compute total taps (entries + exits)
wembley["Date"] = pd.to_datetime(wembley["TravelDate"].astype(str), format="%Y%m%d")
wembley["taps"] = wembley["EntryTapCount"] + wembley["ExitTapCount"]

# Daily total across the 3 stations
daily = wembley.groupby("Date", as_index=False).agg({"taps": "sum"})

# === 3. Add event dummies (Match_t, Concert_t) ===
matches["Date"] = pd.to_datetime(matches["date"], dayfirst=True)
concerts["Date"] = pd.to_datetime(concerts["Concert date"])

match_dates = matches["Date"].unique()
concert_dates = concerts["Date"].unique()

daily["Match_t"] = daily["Date"].isin(match_dates).astype(int)
daily["Concert_t"] = daily["Date"].isin(concert_dates).astype(int)

# === 4. Merge general transport data (TubeJourney, BusJourney) ===
journeys["Date"] = pd.to_datetime(journeys["TravelDate"].astype(str), format="%Y%m%d")

daily = daily.merge(
    journeys[["Date", "DayOfWeek", "TubeJourneyCount", "BusJourneyCount"]],
    on="Date", how="left"
)

# Rename immediately to model variable names
daily = daily.rename(columns={
    "TubeJourneyCount": "TubeJourney",
    "BusJourneyCount": "BusJourney"
})

# Add weekday/weekend info → Weekend_t
daily["DayOfWeek"] = daily["DayOfWeek"].fillna(daily["Date"].dt.day_name())
daily["Weekend_t"] = daily["DayOfWeek"].isin(["Saturday", "Sunday"]).astype(int)

# 5. Create lagged variable for previous-day flow
daily = daily.sort_values("Date")
daily["taps_lag1"] = daily["taps"].shift(1)

# Drop rows where lag is missing (first day)
daily = daily.dropna(subset=["taps_lag1"])

# Take logs to stabilize variance
daily["log_taps"] = np.log(daily["taps"])
daily["log_taps_lag1"] = np.log(daily["taps_lag1"])

# === 6. Run regression ===
# log(taps_t) = β0 + β1 Match_t + β2 Concert_t + β3 log(taps_{t−1})
#               + β4 Weekend_t + β5 TubeJourney + β6 BusJourney + ε_t
formula = """
log_taps ~ Match_t + Concert_t
           + log_taps_lag1
           + Weekend_t
           + TubeJourney
           + BusJourney
"""

model = smf.ols(formula=formula, data=daily).fit()

# === 7. Print summary ===
print(model.summary())


In [ ]:
# === 8. Plot actual vs predicted values ===
daily["pred_log_taps"] = model.predict(daily)

plt.figure(figsize=(6,5))
plt.scatter(daily["log_taps"], daily["pred_log_taps"], alpha=0.5)
plt.xlabel("Actual log(taps)")
plt.ylabel("Predicted log(taps)")
plt.title("Actual vs Predicted Passenger Flows (Wembley Stations)")
plt.tight_layout()
plt.savefig("actual_predicted_flows.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Create a simple event / non-event label
daily["event_type"] = "No event"
daily.loc[daily["Match_t"] == 1, "event_type"] = "Match"
daily.loc[(daily["Concert_t"] == 1) & (daily["Match_t"] == 0), "event_type"] = "Concert"

# Compute average daily taps by event type
avg_taps = (
    daily.groupby("event_type")["taps"]
    .mean()
    .reindex(["No event", "Match", "Concert"])
)

plt.figure()
avg_taps.plot(kind="bar")
plt.ylabel("Average daily taps (entries + exits)")
plt.xlabel("Type of event")   # ← updated X-axis label
#plt.title("Average passenger flows on Wembley stations\nby event type")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("Avg passenger.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Create the event indicator if needed:
daily["Event_t"] = ((daily["Match_t"] + daily["Concert_t"]) > 0).astype(int)

# Aggregate taps by Date, Station and Event_t
wembley["Date"] = pd.to_datetime(wembley["TravelDate"].astype(str), format="%Y%m%d")
wembley["Event_t"] = wembley["Date"].isin(daily[daily["Event_t"] == 1]["Date"]).astype(int)

station_event = (
    wembley
    .groupby(["Station", "Event_t"])["taps"]
    .mean()
    .reset_index()
)

# Replace 0/1 with labels
station_event["event_label"] = station_event["Event_t"].map({0: "No event", 1: "Event"})

# Pivot for easier plotting
pivot_se = station_event.pivot(index="Station", columns="event_label", values="taps")

plt.figure()
pivot_se.plot(kind="bar")
plt.ylabel("Average daily taps")
plt.xlabel("Station")
#plt.title("Average passenger flows by station\nEvent vs non-event days")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("Event_no event.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Copy daily data to avoid modifying original
sim = daily.copy()

# Create Event_t if needed:
sim["Event_t"] = ((sim["Match_t"] + sim["Concert_t"]) > 0).astype(int)

# Assume shuttle service reduces event-day flows by 20%
reduction_share = 0.20

sim["taps_with_shuttle"] = sim["taps"]
sim.loc[sim["Event_t"] == 1, "taps_with_shuttle"] = (
    sim.loc[sim["Event_t"] == 1, "taps"] * (1 - reduction_share)
)

# Compute average taps on event days before/after shuttle
event_only = sim[sim["Event_t"] == 1]

avg_before = event_only["taps"].mean()
avg_after = event_only["taps_with_shuttle"].mean()

before_after = pd.Series(
    {"Before shuttle": avg_before, "With shuttle": avg_after}
)

plt.figure()
before_after.plot(kind="bar")
plt.ylabel("Average daily taps on event days")
plt.xlabel("Scenario")
#plt.title("Simulated impact of shuttle buses\non Wembley station crowding")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("shuttle_notshuttle.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# === 1. Load data ===
footfall = pd.read_csv("StationFootfall_2022_2025.csv")
matches = pd.read_excel("wembley_matches_2021_2025.xlsx")
concerts = pd.read_csv("wembley_concerts_2022_2025.csv")

wembley_stations = ["Wembley Park", "Wembley Central", "North Wembley"]
df = footfall[footfall["Station"].isin(wembley_stations)].copy()

# Convert and aggregate
df["Date"] = pd.to_datetime(df["TravelDate"].astype(str), format="%Y%m%d")
df["taps"] = df["EntryTapCount"] + df["ExitTapCount"]

station_daily = df.groupby(["Date", "Station"], as_index=False)["taps"].sum()
total_daily = (
    station_daily.groupby("Date", as_index=False)["taps"]
    .sum()
    .sort_values("Date")
)

# === 2. Prepare event data ===
matches["Date"] = pd.to_datetime(matches["date"], dayfirst=True)
concerts["Date"] = pd.to_datetime(concerts["Concert date"])

match_dates = set(matches["Date"].dt.date)
concert_dates = set(concerts["Date"].dt.date)
event_dates = match_dates.union(concert_dates)

# Dummy for any event (match or concert)
total_daily["Event_t"] = total_daily["Date"].dt.date.isin(event_dates).astype(int)

# === 3. Detect anomalies ===
mean_taps = total_daily["taps"].mean()
std_taps = total_daily["taps"].std()
threshold = mean_taps + 2 * std_taps
q98 = total_daily["taps"].quantile(0.98)

spikes = total_daily[total_daily["taps"] >= q98]
events_only = total_daily[total_daily["Event_t"] == 1]

# === 4. Plot ===
fig, ax = plt.subplots(figsize=(12, 5))

# Main line — blue
ax.plot(
    total_daily["Date"], total_daily["taps"],
    linewidth=1.3, color="#1f77b4",
    label="Total tap-ins/outs (all Wembley stations)"
)

# Spikes — red
ax.scatter(
    spikes["Date"], spikes["taps"],
    marker="o", s=30, color="#e74c3c",
    label="Detected spike (top 2%)"
)

# Event days — yellow crosses
ax.scatter(
    events_only["Date"], events_only["taps"],
    marker="x", s=40, color="#f1c40f",
    label="Event day (match or concert)"
)

# Threshold line — grey dashed
ax.axhline(
    threshold, linestyle="--", color="#7f8c8d", linewidth=1.2,
    label="Anomaly threshold (mean + 2σ)"
)

# === 5. Formatting ===
#ax.set_title("Wembley transport spikes vs event days (Jan 2022–Dec 2025)", fontsize=13, weight="bold")
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Total taps (entries + exits)", fontsize=11)
ax.legend(loc="upper left", fontsize=9)

# Month formatting: every 2 months
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

# Limit x-axis range
ax.set_xlim(pd.Timestamp("2022-01-01"), pd.Timestamp("2025-12-31"))

# Grid and layout
ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.7)
fig.autofmt_xdate(rotation=0)

plt.tight_layout()
plt.savefig("wembley_spikes_2022_2025.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Graph 2
station_daily = station_daily.sort_values(["Station", "Date"])

station_daily["baseline"] = (
    station_daily
    .groupby("Station")["taps"]
    .transform(lambda s: s.rolling(window=14, min_periods=7, center=True).median())
)

# Dummy for event day
station_daily["Event_t"] = station_daily["Date"].dt.date.isin(event_dates).astype(int)

event_station = station_daily[station_daily["Event_t"] == 1].copy()
event_station["delta_pct"] = 100 * (event_station["taps"] - event_station["baseline"]) / event_station["baseline"]
event_station = event_station.sort_values("Date")

fig, ax = plt.subplots(figsize=(12, 5))

# wider bars and improved colors
color_map = {
    "North Wembley": "#1f77b4",
    "Wembley Central": "#ff7f0e",
    "Wembley Park": "#2ca02c"
}

for station, sub in event_station.groupby("Station"):
    ax.bar(
        sub["Date"], sub["delta_pct"],
        width=8, color=color_map.get(station, None),
        label=station, alpha=0.9, align="center"
    )

# Format x-axis to show months and years
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

# Titles and labels
#ax.set_title("Traffic spikes vs 14-day baseline on event days (Wembley stations, 2022–2025)", fontsize=13, weight="bold")
ax.set_ylabel("Δ taps (%) relative to 14-day median", fontsize=11)
ax.axhline(0, color="gray", linewidth=0.8)
ax.legend(title="Station", fontsize=9)
ax.grid(True, linestyle=":", linewidth=0.5, axis="y", alpha=0.6)

plt.tight_layout()
plt.savefig("traffic_spikes_2022_2025.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Copy original daily dataset
sim = daily.copy()

# Dummy for any event (if not already created)
sim["Event_t"] = ((sim["Match_t"] + sim["Concert_t"]) > 0).astype(int)

# Add dummy for shuttle scenario (shuttles only on event days)
sim["Shuttle_t"] = sim["Event_t"]   # 1 on event days, 0 otherwise

# Apply reduction of 20% passenger load on event days
reduction_share = 0.2
sim["taps_with_shuttle"] = sim["taps"]
sim.loc[sim["Event_t"] == 1, "taps_with_shuttle"] = (
    sim.loc[sim["Event_t"] == 1, "taps"] * (1 - reduction_share)
)

# Log-transform new variable
sim["log_taps_shuttle_t"] = np.log(sim["taps_with_shuttle"])

# Build regression with Shuttle_t as additional variable
formula_shuttle = """
log_taps_shuttle_t ~ Match_t + Concert_t
                     + log_taps_lag1
                     + Weekend_t
                     + TubeJourney
                     + BusJourney
                     + Shuttle_t
"""

model_shuttle = smf.ols(formula=formula_shuttle, data=sim).fit()
print(model_shuttle.summary())

# Compare predicted passenger flows
# model — the baseline model log_taps ~ Match_t + ... + BusJourney
sim["pred_log_no_shuttle"] = model.predict(sim)
sim["pred_log_with_shuttle"] = model_shuttle.predict(sim)

# Convert back from logs to actual passenger counts
sim["pred_taps_no_shuttle"] = np.exp(sim["pred_log_no_shuttle"])
sim["pred_taps_with_shuttle"] = np.exp(sim["pred_log_with_shuttle"])

# Visual comparison for event days
event_only = sim[sim["Event_t"] == 1][["Date", "pred_taps_no_shuttle", "pred_taps_with_shuttle"]]
avg_no_shuttle = event_only["pred_taps_no_shuttle"].mean()
avg_with_shuttle = event_only["pred_taps_with_shuttle"].mean()

print(f"Average predicted flow on event days without shuttles: {avg_no_shuttle:,.0f}")
print(f"Average predicted flow on event days with shuttles: {avg_with_shuttle:,.0f}")


In [ ]:
!pip install statsmodels
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# 1. Load datasets
footfall = pd.read_csv("StationFootfall_2022_2025.csv")
matches = pd.read_excel("wembley_matches_2021_2025.xlsx")
concerts = pd.read_csv("wembley_concerts_2022_2025.csv")
journeys = pd.read_csv("Journeys_2024_2025 .csv")

# 2. Filter Wembley stations
wembley_stations = ["Wembley Park", "Wembley Central", "North Wembley"]
wembley = footfall[footfall["Station"].isin(wembley_stations)].copy()

# Convert date and compute total taps (entries + exits)
wembley["Date"] = pd.to_datetime(wembley["TravelDate"].astype(str), format="%Y%m%d")
wembley["taps"] = wembley["EntryTapCount"] + wembley["ExitTapCount"]

# Daily total across the 3 stations
daily = wembley.groupby("Date", as_index=False).agg({"taps": "sum"})

# === 3. Add event dummies (Match_t, Concert_t) ===
matches["Date"] = pd.to_datetime(matches["date"], dayfirst=True)
concerts["Date"] = pd.to_datetime(concerts["Concert date"])

match_dates = matches["Date"].unique()
concert_dates = concerts["Date"].unique()

daily["Match_t"]   = daily["Date"].isin(match_dates).astype(int)
daily["Concert_t"] = daily["Date"].isin(concert_dates).astype(int)

# === 4. Merge general transport data (TubeJourney, BusJourney) ===
journeys["Date"] = pd.to_datetime(journeys["TravelDate"].astype(str), format="%Y%m%d")

daily = daily.merge(
    journeys[["Date", "DayOfWeek", "TubeJourneyCount", "BusJourneyCount"]],
    on="Date", how="left"
)

# Rename to model variable names
daily = daily.rename(columns={
    "TubeJourneyCount": "TubeJourney",
    "BusJourneyCount": "BusJourney"
})

# Add weekday/weekend info → Weekend_t
daily["DayOfWeek"] = daily["DayOfWeek"].fillna(daily["Date"].dt.day_name())
daily["Weekend_t"] = daily["DayOfWeek"].isin(["Saturday", "Sunday"]).astype(int)

# *** NEW VARIABLE: effect "weekend match" ***
daily["WeekendMatch_t"] = daily["Weekend_t"] * daily["Match_t"]

# 5. Create lagged variable for previous-day flow
daily = daily.sort_values("Date")
daily["taps_lag1"] = daily["taps"].shift(1)

# Drop rows where lag is missing (first day)
daily = daily.dropna(subset=["taps_lag1"])

# Take logs to stabilize variance
daily["log_taps"] = np.log(daily["taps"])
daily["log_taps_lag1"] = np.log(daily["taps_lag1"])

# === 6. Run regression ===
# log(taps_t) = β0 + β1 Match_t + β2 Concert_t + β3 log(taps_{t−1})
#             + β4 Weekend_t + β5 TubeJourney + β6 BusJourney
#             + β7 WeekendMatch_t + ε_t
formula = """
log_taps ~ Match_t + Concert_t
           + log_taps_lag1
           + Weekend_t
           + TubeJourney
           + BusJourney
           + WeekendMatch_t
"""

model = smf.ols(formula=formula, data=daily).fit()

# === 7. Print summary ===
print(model.summary())
